In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchsummary import summary
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np
import time

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size ,patch_size ,num_hiddens):
        super().__init__()
        self.num_patches= (img_size// patch_size ) ** 2
        self.conv = nn.Conv2d(3,num_hiddens,kernel_size=patch_size,stride=patch_size)

    def forward(self,x):
        return self.conv(x).flatten(2).transpose(1,2)


class ViTMLP(nn.Module):
    def __init__(self,mlp_num_hidden, mlp_num_outputs,dropout=0.5):
        super().__init__()
        self.dense1=nn.LazyLinear(mlp_num_hidden)
        self.gelu = nn.GELU()
        self.dropout1=nn.Dropout(dropout)
        self.dense2=nn.LazyLinear(mlp_num_outputs)
        self.dropout2=nn.Dropout(dropout)

    def forward(self,x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(self.dense1(x)))))


class ViTBlock(nn.Module):
    def __init__(self,num_hidden,norm_shape,mlp_num_hiddens,num_heads,dropout,use_bias=False):
        super().__init__()
        self.ln1= nn.LayerNorm(norm_shape)
        self.attention = nn.MultiheadAttention(num_hidden,num_heads,dropout,use_bias,batch_first=True)
        self.ln2= nn.LayerNorm(norm_shape)
        self.mlp= ViTMLP(mlp_num_hiddens,num_hidden,dropout)

    def forward(self,x):
        x2= self.ln1(x)
        attention_output,_=self.attention(x2,x2,x2)
        x = x + attention_output
        x2= self.ln2(x)
        mlp_output= self.mlp(x2)
        x = x + mlp_output
        return x



In [18]:
class ViT(nn.Module):
    def __init__(self, eta, n_iter, random_state, batch_size, img_size, patch_size,
                 num_hidden, norm_shape, mlp_num_hiddens, num_heads, num_blks,
                 num_classes=100, dropout=0.2):
        super().__init__()
        self.eta = eta
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state

        self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hidden)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hidden))
        num_steps = self.patch_embedding.num_patches + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hidden))
        self.drop_out = nn.Dropout(dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(num_hidden, norm_shape, mlp_num_hiddens,
                                                    num_heads, dropout, use_bias=False))
        self.head = nn.Sequential(nn.LayerNorm(num_hidden), nn.Linear(num_hidden, num_classes))

        self.device = torch.device(
            "cuda" if torch.cuda.is_available()
            else "cpu"
        )
        print(f"Using device: {self.device}")

        self.train_losses_     = []
        self.train_accuracies_ = []
        self.val_losses_       = []
        self.val_accuracies_   = []
        self.to(self.device)

    def forward(self, x):
        x = self.patch_embedding(x)
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), 1)
        x = self.drop_out(x + self.pos_embedding)
        for blk in self.blks:
            x = blk(x)
        x = self.head(x[:, 0])
        return x

    def iter_mini_batch(self,X,y):
        dataset=TensorDataset(X,y)
        dataloader= DataLoader(dataset,batch_size=self.batch_size,shuffle=True)
        return dataloader

    def fit(self, X,y,X_test,y_test):
        torch.manual_seed(self.random_state)
        start_time = time.time()
        optimizer = optim.Adam(self.parameters(), lr=self.eta)
        criterion= nn.CrossEntropyLoss()
        for epoch in range(self.n_iter):
            epoch_start= time.time()
            self.train()
            train_loader = self.iter_mini_batch(X,y)
            running_loss=0.0
            correct=0
            total =0

            for X_batch,y_batch in train_loader:
                X_batch= X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                optimizer.zero_grad()
                y_hat= self(X_batch)
                loss= criterion(y_hat,y_batch)
                loss.backward()
                optimizer.step()

                running_loss+=loss.item()
                preds=y_hat.argmax(dim=1)
                correct+= (preds == y_batch).sum().item()
                total += y_batch.numel()
            epoch_time = time.time()-epoch_start
            epoch_loss = running_loss/(len(train_loader))
            epoch_acc= 100 * correct/total

            self.train_losses_.append(epoch_loss)
            self.train_accuracies_.append(epoch_acc)

            val_loss, val_acc = self._evaluate(X_test, y_test, criterion)
            self.val_losses_.append(val_loss)
            self.val_accuracies_.append(val_acc)
            current_lr = optimizer.param_groups[0]['lr']

            print(f'Epoch {epoch+1:>3}/{self.n_iter} | '
                  f'Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}% | '
                  f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% |'
                  f'lr: {current_lr:.6f} |'
                  f'Time: {epoch_time:.1f}s')

        self.training_time_ = time.time() - start_time    #stop the clock
        mins = self.training_time_ // 60
        secs = self.training_time_ % 60
        print(f"\nTraining complete: {int(mins)}m {secs:.1f}s")
        return self

    def _evaluate(self, X_test, y_test, criterion):
        test_loader=self.iter_mini_batch(X_test,y_test)
        self.eval()
        running_loss = 0.0
        correct      = 0
        total        = 0
        with torch.no_grad():
            for xin, target in test_loader:
                xin = xin.to(self.device)
                target = target.to(self.device)

                outputs = self(xin)
                loss=criterion(outputs.view(-1, outputs.size(-1)), target.view(-1))

                running_loss+= loss.item()
                predicted = outputs.argmax(dim=-1)
                total += target.numel()
                correct += (predicted == target).sum().item()
        val_loss = running_loss / len(test_loader)
        val_acc  = 100.0 * correct / total
        return val_loss, val_acc

    def predict(self, X):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)
        self.eval()
        with torch.no_grad():
            X        = X.to(self.device)
            logits   = self(X)
            _, preds = torch.max(logits, dim=1)
        return preds.cpu().numpy()
    def accuracy(self, X, y):
        if isinstance(y, torch.Tensor):
            y = y.numpy()
        preds = self.predict(X)
        return 100.0 * np.mean(preds == y)
    def training_summary(self):
      mins = int(self.training_time_ // 60)
      secs = self.training_time_ % 60
      params = sum(p.numel() for p in self.parameters())
      print(f"Parameters:    {params:,}")
      print(f"Training time: {mins}m {secs:.1f}s")
      print(f"Best val acc:  {max(self.val_accuracies_):.2f}%")
      print(f"Final test acc: call model.accuracy(X_test, y_test)")




In [ ]:
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD  = (0.2673, 0.2564, 0.2762)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_set = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

print(f"Train size: {len(train_set)}, Test size: {len(test_set)}")
print(f"Image shape: {train_set[0][0].shape}")

100%|██████████| 169M/169M [51:55<00:00, 54.2kB/s]


Train size: 50000, Test size: 10000
Image shape: torch.Size([3, 32, 32])


In [10]:
!pip install torchinfo -q
from torchinfo import summary

def get_complexity_stats(model, input_size=(1, 3, 32, 32)):
    stats = summary(model, input_size=input_size, verbose=0)
    params = stats.total_params
    macs = stats.total_mult_adds
    flops = macs * 2
    return params, flops

In [ ]:
vit_configs = [
    {'name': 'C1_baseline', 'patch_size': 4, 'num_hidden': 256, 'num_blks': 4, 'num_heads': 4},
    {'name': 'C2_patch8', 'patch_size': 8, 'num_hidden': 256, 'num_blks': 4,'num_heads': 4},
    {'name': 'C3_embed512','patch_size': 4, 'num_hidden': 512,'num_blks': 4, 'num_heads': 4},
    {'name': 'C4_blocks8', 'patch_size': 4, 'num_hidden': 256,'num_blks': 8,'num_heads': 4},
    {'name': 'C5_heads8', 'patch_size': 4, 'num_hidden': 256,'num_blks': 4, 'num_heads': 8},
    {'name': 'C6_max', 'patch_size': 8, 'num_hidden': 512, 'num_blks': 8,'num_heads': 8},
    {'name': 'C7_patch8_wide','patch_size': 8, 'num_hidden': 512, 'num_blks': 4, 'num_heads': 4},
    {'name': 'C8_deep_wide', 'patch_size': 4, 'num_hidden': 512, 'num_blks': 8, 'num_heads': 8},
]

In [ ]:
def dataset_to_tensors(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    X, y = next(iter(loader))
    return X, y

X_train, y_train = dataset_to_tensors(train_set)
X_test, y_test = dataset_to_tensors(test_set)

print(X_train.shape, y_train.shape)   # (50000, 3, 32, 32), (50000,)
print(X_test.shape, y_test.shape)     # (10000, 3, 32, 32), (10000,)

torch.Size([50000, 3, 32, 32]) torch.Size([50000])
torch.Size([10000, 3, 32, 32]) torch.Size([10000])


In [ ]:

import pickle

vit_results = []

for cfg in vit_configs:
    print(f"\n{'='*70}\nTraining {cfg['name']}: {cfg}\n{'='*70}")

    model = ViT(
        eta=0.001, n_iter=10, batch_size=64, random_state=42,
        img_size=32, patch_size=cfg['patch_size'],
        num_hidden=cfg['num_hidden'],
        norm_shape=cfg['num_hidden'],
        mlp_num_hiddens=cfg['num_hidden'] * 4,
        num_heads=cfg['num_heads'],
        num_blks=cfg['num_blks'],
        num_classes=100,
        dropout=0.2,
    )

    params, flops = get_complexity_stats(model)

    model.fit(X_train, y_train, X_test, y_test)   # adjust to match your fit()'s actual signature

    result = {
        'config': cfg['name'],
        **cfg,
        'params': params,
        'flops': flops,
        'final_train_acc': model.train_accuracies_[-1],
        'final_val_acc': model.val_accuracies_[-1],
        'best_val_acc': max(model.val_accuracies_),
        'training_time_s': model.training_time_,
        'train_losses': model.train_losses_,
        'val_losses': model.val_losses_,
        'val_accuracies': model.val_accuracies_,
    }
    vit_results.append(result)

    # checkpoint after every single config — never lose completed work to a later crash
    with open('vit_sweep_results.pkl', 'wb') as f:
        pickle.dump(vit_results, f)
    print(f"Checkpointed after {cfg['name']}")

import pandas as pd
vit_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('train_losses', 'val_losses', 'val_accuracies')} for r in vit_results])
vit_df


Training C1_baseline: {'name': 'C1_baseline', 'patch_size': 4, 'num_hidden': 256, 'num_blks': 4, 'num_heads': 4}
Using device: cuda
Epoch   1/10 | Loss: 3.9621 | Acc: 8.71% | Val Loss: 3.6993 | Val Acc: 12.63% |lr: 0.001000 |Time: 30.3s
Epoch   2/10 | Loss: 3.6979 | Acc: 12.20% | Val Loss: 3.5642 | Val Acc: 14.27% |lr: 0.001000 |Time: 29.4s
Epoch   3/10 | Loss: 3.6024 | Acc: 14.04% | Val Loss: 3.5091 | Val Acc: 16.00% |lr: 0.001000 |Time: 30.5s
Epoch   4/10 | Loss: 3.5374 | Acc: 15.00% | Val Loss: 3.4262 | Val Acc: 17.60% |lr: 0.001000 |Time: 29.9s
Epoch   5/10 | Loss: 3.4796 | Acc: 16.31% | Val Loss: 3.3778 | Val Acc: 17.83% |lr: 0.001000 |Time: 29.7s
Epoch   6/10 | Loss: 3.4735 | Acc: 16.62% | Val Loss: 3.4056 | Val Acc: 17.59% |lr: 0.001000 |Time: 29.7s
Epoch   7/10 | Loss: 3.4713 | Acc: 16.31% | Val Loss: 3.3741 | Val Acc: 18.64% |lr: 0.001000 |Time: 29.7s
Epoch   8/10 | Loss: 3.4670 | Acc: 16.53% | Val Loss: 3.4024 | Val Acc: 18.16% |lr: 0.001000 |Time: 29.7s
Epoch   9/10 | Loss:

,config,name,patch_size,num_hidden,num_blks,num_heads,params,flops,final_train_acc,final_val_acc,best_val_acc,training_time_s
0,C1_baseline,C1_baseline,4,256,4,4,3210596,5870792,17.712,19.40,19.61,318.258148
1,C2_patch8,C2_patch8,8,256,4,4,3235172,5846216,16.678,19.98,19.98,108.058039
2,C3_embed512,C3_embed512,4,512,4,4,12712548,20129992,4.638,5.15,8.43,918.888749
3,C4_blocks8,C4_blocks8,4,256,8,4,6365540,10083528,5.850,6.37,12.02,624.006371
4,C5_heads8,C5_heads8,4,256,4,8,3210596,5870792,25.090,26.15,26.15,336.946744
5,C6_max,C6_max,8,512,8,8,25363044,36894920,6.076,6.82,9.86,560.840470
6,C7_patch8_wide,C7_patch8_wide,8,512,4,4,12761700,20080840,6.148,6.91,7.50,277.635310
7,C8_deep_wide,C8_deep_wide,4,512,8,8,25313892,36944072,4.082,4.23,10.20,1872.753355


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels,kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv2d(in_channels, out_channels,kernel_size=1, stride=stride, bias=False),
                            nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, eta=0.001, n_iter=100, batch_size=128, random_state=42, num_classes=10):
        super().__init__()
        self.eta = eta
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state
        self.num_classes = num_classes

        self.device = torch.device('cuda' if torch.cuda.is_available() else
                                   'mps' if torch.backends.mps.is_available() else
                                   'cpu')
        print(f'Using device: {self.device}')

        self.stem= nn.Sequential(
            nn.Conv2d(in_channels=3,out_channels=64,kernel_size=3,padding=1,stride=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.module1= nn.Sequential(
            ResidualBlock(64,64,stride=1),
            ResidualBlock(64,64,stride=1)
        )

        self.module2= nn.Sequential(
            ResidualBlock(64,128,stride=2),
            ResidualBlock(128,128,stride=1)
        )

        self.module3= nn.Sequential(
            ResidualBlock(128,256,stride=2),
            ResidualBlock(256,256,stride=1)
        )

        self.module4= nn.Sequential(
            ResidualBlock(256,512,stride=2),
            ResidualBlock(512,512,stride=1)
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

        self.train_losses_     = []
        self.train_accuracies_ = []
        self.val_losses_       = []
        self.val_accuracies_   = []

        self.to(self.device)
        print(f"ResNet-18 initialised.")

    def forward(self,x):
        x = self.stem(x)
        x = self.module1(x)
        x = self.module2(x)
        x = self.module3(x)
        x = self.module4(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

    def iter_minibatch(self, X, y):
        dataset    = TensorDataset(X, y)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        return dataloader

    def _evaluate(self, X, y, criterion):
        loader = self.iter_minibatch(X, y)
        self.eval()

        running_loss = 0.0
        correct      = 0
        total        = 0

        with torch.no_grad():
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                outputs = self(X_batch)
                loss    = criterion(outputs, y_batch)

                running_loss += loss.item()
                _, predicted  = torch.max(outputs, dim=1)
                total        += y_batch.size(0)
                correct      += (predicted == y_batch).sum().item()

        avg_loss = running_loss / len(loader)
        accuracy = 100.0 * correct / total
        return avg_loss, accuracy

    def predict(self, X):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)
        self.eval()
        with torch.no_grad():
            X        = X.to(self.device)
            logits   = self(X)
            _, preds = torch.max(logits, dim=1)
        return preds.cpu().numpy()
    def accuracy(self, X, y):
        if isinstance(y, torch.Tensor):
            y = y.numpy()
        preds = self.predict(X)
        return 100.0 * np.mean(preds == y)

    def fit(self, X, y, X_val, y_val):
        torch.manual_seed(self.random_state)
        start_time = time.time()
        optimizer = optim.Adam(self.parameters(), lr=self.eta)
        criterion    = nn.CrossEntropyLoss()
        train_loader = self.iter_minibatch(X, y)

        for epoch in range(self.n_iter):
            epoch_start = time.time()
            self.train()

            running_loss = 0.0
            correct      = 0
            total        = 0

            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                optimizer.zero_grad()
                y_hat = self(X_batch)
                loss  = criterion(y_hat, y_batch)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                preds = y_hat.argmax(dim=1)
                correct+= (preds == y_batch).sum().item()
                total += y_batch.size(0)

            epoch_time = time.time() - epoch_start
            epoch_loss = running_loss / len(train_loader)
            epoch_acc  = 100.0 * correct / total

            self.train_losses_.append(epoch_loss)
            self.train_accuracies_.append(epoch_acc)

            val_loss, val_acc = self._evaluate(X_val, y_val, criterion)
            self.val_losses_.append(val_loss)
            self.val_accuracies_.append(val_acc)
            current_lr = optimizer.param_groups[0]['lr']

            print(f'Epoch {epoch+1:>3}/{self.n_iter} | '
                  f'Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}% | '
                  f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% |'
                  f'lr: {current_lr:.6f} |'
                  f'Time: {epoch_time:.1f}s')


        self.training_time_ = time.time() - start_time    #stop the clock
        mins = self.training_time_ // 60
        secs = self.training_time_ % 60
        print(f"\nTraining complete: {int(mins)}m {secs:.1f}s")
        return self
    def training_summary(self):
      mins = int(self.training_time_ // 60)
      secs = self.training_time_ % 60
      params = sum(p.numel() for p in self.parameters())
      print(f"Parameters:    {params:,}")
      print(f"Training time: {mins}m {secs:.1f}s")
      print(f"Best val acc:  {max(self.val_accuracies_):.2f}%")
      print(f"Final test acc: call model.accuracy(X_test, y_test)")

In [ ]:
resnet = ResNet18(eta=0.001, n_iter=10, batch_size=64, random_state=42, num_classes=100)
resnet_params, resnet_flops = get_complexity_stats(resnet)
print(f"ResNet-18 params: {resnet_params:,}, FLOPs: {resnet_flops:,}")

resnet.fit(X_train, y_train, X_test, y_test)

resnet_result = {
    'config': 'ResNet18',
    'params': resnet_params, 'flops': resnet_flops,
    'final_train_acc': resnet.train_accuracies_[-1],
    'final_val_acc': resnet.val_accuracies_[-1],
    'best_val_acc': max(resnet.val_accuracies_),
    'training_time_s': resnet.training_time_,
    'train_losses': resnet.train_losses_,
    'val_losses': resnet.val_losses_,
    'val_accuracies': resnet.val_accuracies_,
}



Using device: cuda
ResNet-18 initialised.
ResNet-18 params: 11,220,196, FLOPs: 1,111,088,072
Epoch   1/10 | Loss: 3.5884 | Acc: 14.69% | Val Loss: 3.0888 | Val Acc: 24.74% |lr: 0.001000 |Time: 43.2s
Epoch   2/10 | Loss: 2.6359 | Acc: 31.82% | Val Loss: 2.3489 | Val Acc: 37.78% |lr: 0.001000 |Time: 43.1s
Epoch   3/10 | Loss: 2.0554 | Acc: 44.31% | Val Loss: 2.0573 | Val Acc: 44.00% |lr: 0.001000 |Time: 42.7s
Epoch   4/10 | Loss: 1.6610 | Acc: 53.25% | Val Loss: 1.8294 | Val Acc: 49.17% |lr: 0.001000 |Time: 42.6s
Epoch   5/10 | Loss: 1.3418 | Acc: 61.07% | Val Loss: 1.8219 | Val Acc: 50.50% |lr: 0.001000 |Time: 42.6s
Epoch   6/10 | Loss: 1.0243 | Acc: 69.22% | Val Loss: 1.7205 | Val Acc: 54.37% |lr: 0.001000 |Time: 42.8s
Epoch   7/10 | Loss: 0.7137 | Acc: 78.00% | Val Loss: 1.7925 | Val Acc: 54.75% |lr: 0.001000 |Time: 42.6s
Epoch   8/10 | Loss: 0.4427 | Acc: 86.07% | Val Loss: 1.9506 | Val Acc: 54.82% |lr: 0.001000 |Time: 42.6s
Epoch   9/10 | Loss: 0.2636 | Acc: 91.73% | Val Loss: 2.242

In [11]:
import os
from transformers import AutoImageProcessor, SwinForImageClassification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/cifar100_data'   # directory, not the .gz file itself
os.makedirs(DATA_ROOT, exist_ok=True)

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
processor = AutoImageProcessor.from_pretrained("microsoft/swin-tiny-patch4-window7-224")
print(processor.image_mean, processor.image_std, processor.size)

swin_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

(0.485, 0.456, 0.406) (0.229, 0.224, 0.225) SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)


In [13]:
train_set_224 = torchvision.datasets.CIFAR100(root=DATA_ROOT, train=True, download=True, transform=swin_transform)
test_set_224  = torchvision.datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=swin_transform)
print(f"Train: {len(train_set_224)}, Test: {len(test_set_224)}")

Train: 50000, Test: 10000


In [14]:
def load_swin_frozen(model_name, num_classes=100):
    model = SwinForImageClassification.from_pretrained(
        model_name,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )
    for param in model.swin.parameters():
        param.requires_grad = False
    return model

def report_params(model, name):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"{name} — Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.2f}%)")
    return trainable, total

In [15]:
def finetune_swin(model, train_set, test_set, n_iter=5, batch_size=32, lr=2e-5, device='cuda'):
    model = model.to(device)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, train_accs, val_losses, val_accs, epoch_times = [], [], [], [], []
    start_time = time.time()

    for epoch in range(n_iter):
        epoch_start = time.time()
        model.train()
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
        running_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=X_batch).logits
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            correct += (outputs.argmax(dim=1) == y_batch).sum().item()
            total += y_batch.size(0)

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        train_losses.append(running_loss / len(train_loader))
        train_accs.append(100 * correct / total)

        model.eval()
        val_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(pixel_values=X_batch).logits
                val_loss += criterion(outputs, y_batch).item()
                val_correct += (outputs.argmax(dim=1) == y_batch).sum().item()
                val_total += y_batch.size(0)

        val_losses.append(val_loss / len(val_loader))
        val_accs.append(100 * val_correct / val_total)

        print(f'Epoch {epoch+1}/{n_iter} | Loss: {train_losses[-1]:.4f} | Acc: {train_accs[-1]:.2f}% | '
              f'Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_accs[-1]:.2f}% | Time: {epoch_time:.1f}s')

    total_time = time.time() - start_time
    print(f"Training complete: {int(total_time//60)}m {total_time%60:.1f}s")
    return {'train_losses': train_losses, 'train_accs': train_accs,
            'val_losses': val_losses, 'val_accs': val_accs,
            'epoch_times': epoch_times, 'total_time': total_time}

In [16]:
swin_tiny = load_swin_frozen("microsoft/swin-tiny-patch4-window7-224")
report_params(swin_tiny, "Swin-Tiny")

swin_tiny_result = finetune_swin(swin_tiny, train_set_224, test_set_224,
                                   n_iter=5, batch_size=32, lr=2e-5, device=device)

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Swin-Tiny — Trainable: 76,900 / Total: 27,596,254 (0.28%)
Epoch 1/5 | Loss: 4.0572 | Acc: 23.46% | Val Loss: 3.4956 | Val Acc: 46.41% | Time: 230.0s
Epoch 2/5 | Loss: 3.0592 | Acc: 53.79% | Val Loss: 2.6649 | Val Acc: 58.39% | Time: 228.4s
Epoch 3/5 | Loss: 2.3785 | Acc: 61.23% | Val Loss: 2.1248 | Val Acc: 62.69% | Time: 229.0s
Epoch 4/5 | Loss: 1.9450 | Acc: 64.47% | Val Loss: 1.7882 | Val Acc: 64.74% | Time: 229.0s
Epoch 5/5 | Loss: 1.6727 | Acc: 66.45% | Val Loss: 1.5740 | Val Acc: 66.39% | Time: 229.2s
Training complete: 22m 54.3s


In [17]:
vit_scratch = ViT(
    eta=0.001, n_iter=5, batch_size=32, random_state=42,
    img_size=224, patch_size=16,
    num_hidden=256, norm_shape=256, mlp_num_hiddens=1024,
    num_heads=4, num_blks=4, num_classes=100, dropout=0.2,
)

vit_train_loader_check = DataLoader(train_set_224, batch_size=32, shuffle=False)
Xb, yb = next(iter(vit_train_loader_check))
print(Xb.shape, yb.shape)   # sanity check: (32, 3, 224, 224), (32,)

# ViT.fit expects full tensors, not a Dataset — pull it in batched via the loader
# instead of dataset_to_tensors, since 224x224 is too large to hold as one tensor
def dataset_to_tensors_batched(dataset, batch_size=500):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    xs, ys = [], []
    for xb, yb in loader:
        xs.append(xb)
        ys.append(yb)
    return torch.cat(xs), torch.cat(ys)

NameError: name 'ViT' is not defined